In [ ]:
# [1. 부모 클래스: 기본 센서]
class SensorReading:
    ALERT_T = 35  # [개념 1: 클래스 변수] - 본사 공용 칠판 (모든 센서가 공유)

    # [개념 2: __new__] - C언어 조상님(object)에게 부탁해 빈 깡통 먼저 파내기
    def __new__(cls, *args, **kwargs):
        print(f"\n[1단계 __new__] '{cls.__name__}' 설계도로 메모리에 빈 깡통을 파냅니다.")
        instance = super().__new__(cls)
        print(f"       -> 갓 태어난 깡통 속 딕셔너리: {instance.__dict__} (완전 비어있음!)")
        return instance  # 이 깡통이 그대로 __init__의 self로 배달됨

    # [개념 3: __init__ & self] - 깡통(self)에 데이터(가구) 채워 넣기
    def __init__(self, node_id, t):
        print(f"[2단계 __init__] 태어난 깡통(self)에 데이터를 한 줄씩 꽂습니다.")
        self.node_id = node_id
        print(f"       -> node_id 넣은 직후: {self.__dict__}")
        self.t = t
        print(f"       -> t 넣은 직후: {self.__dict__}")

    # [개념 4: __repr__] - print 찍었을 때 외계어 주소값 대신 예쁘게 출력
    def __repr__(self):
        return f"Reading({self.node_id}, T={self.t})"

    # [개념 5: 일반 메서드] - 본사 매뉴얼에 보관된 판정 기술
    def is_alert(self):
        # self.ALERT_T는 내 주머니에 없어서 본사 칠판(클래스 변수)으로 찾아 올라감
        return self.t > self.ALERT_T


# [2. 자식 클래스: 상속과 super()]
class CalibratedReading(SensorReading):
    def __init__(self, node_id, t, offset):
        # [개념 6: super().__init__] - 부모 공장에 이 깡통(self)을 보내서 기본 조립 먼저 해오기!
        print("[자식 __init__] 부모님(__init__)에게 기본 조립을 먼저 부탁합니다.")
        super().__init__(node_id, t)

        # 내 고유 부품 장착
        self.offset = offset
        print(f"       -> offset까지 장착 완료: {self.__dict__}")

    # [개념 7: 오버라이드] - 부모 기술 대신 내 새 판정 기술로 덮어쓰기
    def is_alert(self):
        return (self.t + self.offset) > self.ALERT_T

In [ ]:
# 1번 센서 탄생
r1 = SensorReading("ESP32-01", 25.3)


[1단계 __new__] 'SensorReading' 설계도로 메모리에 빈 깡통을 파냅니다.
       -> 갓 태어난 깡통 속 딕셔너리: {} (완전 비어있음!)
[2단계 __init__] 태어난 깡통(self)에 데이터를 한 줄씩 꽂습니다.
       -> node_id 넣은 직후: {'node_id': 'ESP32-01'}
       -> t 넣은 직후: {'node_id': 'ESP32-01', 't': 25.3}


In [ ]:
# 둘 다 실행해보기
print("\n--- 메서드 호출 비교 ---")
print("1. 축약형 문법:", r1.is_alert())
print("2. 진짜 정석 문법:", SensorReading.is_alert(r1))


--- 메서드 호출 비교 ---
1. 축약형 문법: False
2. 진짜 정석 문법: False


In [ ]:
r2 = SensorReading("ESP32-02", 36.9)

print("\n--- __repr__ 확인 ---")
print(r1)  # Reading(ESP32-01, T=25.3)  <-- 외계어 주소값 안 나옴!

print("\n--- 클래스 변수 전파 확인 ---")
print("변경 전 경고 여부:", r1.is_alert(), r2.is_alert())  # False True
SensorReading.ALERT_T = 20  # 본사 칠판 기준을 20도로 낮춤
print("변경 후 경고 여부:", r1.is_alert(), r2.is_alert())  # True True (모두에게 전파!)[cite: 1]
SensorReading.ALERT_T = 35  # 원상 복구[cite: 1]


[1단계 __new__] 'SensorReading' 설계도로 메모리에 빈 깡통을 파냅니다.
       -> 갓 태어난 깡통 속 딕셔너리: {} (완전 비어있음!)
[2단계 __init__] 태어난 깡통(self)에 데이터를 한 줄씩 꽂습니다.
       -> node_id 넣은 직후: {'node_id': 'ESP32-02'}
       -> t 넣은 직후: {'node_id': 'ESP32-02', 't': 36.9}

--- __repr__ 확인 ---
Reading(ESP32-01, T=25.3)

--- 클래스 변수 전파 확인 ---
변경 전 경고 여부: False True
변경 후 경고 여부: True True


In [ ]:
print("\n--- 상속과 super() 확인 ---")
c1 = CalibratedReading("ESP32-03", 34.0, 2.0)
print(c1)  # 부모의 __repr__을 그대로 물려받아 출력됨
print("보정 센서 경고 여부:", c1.is_alert())  # True (34.0 + 2.0 = 36.0 > 35)


--- 상속과 super() 확인 ---

[1단계 __new__] 'CalibratedReading' 설계도로 메모리에 빈 깡통을 파냅니다.
       -> 갓 태어난 깡통 속 딕셔너리: {} (완전 비어있음!)
[자식 __init__] 부모님(__init__)에게 기본 조립을 먼저 부탁합니다.
[2단계 __init__] 태어난 깡통(self)에 데이터를 한 줄씩 꽂습니다.
       -> node_id 넣은 직후: {'node_id': 'ESP32-03'}
       -> t 넣은 직후: {'node_id': 'ESP32-03', 't': 34.0}
       -> offset까지 장착 완료: {'node_id': 'ESP32-03', 't': 34.0, 'offset': 2.0}
Reading(ESP32-03, T=34.0)
보정 센서 경고 여부: True
